# ASTR 457 midterm: solutions

Here's one complete solution. I ran it on an example data set that was made the same way as yours but doesn't belong to anyone in the class, so every number below is for that file (it's in the `data/example/` folder next to this notebook). Your numbers will be different because your data are different, so check your own numbers against your own files. The questions and the code layout are the ones in the original notebook. Wherever the original said "your work here" or "your answer goes here", the work and the answer are filled in.

Three questions run through the whole midterm, and it helps to read with them in mind. Which error bar is the right one (Part 1, and again in 2c)? Does the model need the extra parameter (2d)? Could the check I ran have failed (the simulation in 1c can't, the one in 2e can and does, and the "validation" in 3a can't)?

Each answer ends with a short list of where the points come from. Every random cell has a fixed seed, so you get the same numbers when you run the notebook from the top ("Restart & Run All"), which takes about a minute.

# ASTR 457: The Midterm

Posted Thu Oct 8, due Wed Oct 14 by Noon (fork → PR, as always). Graded like a lab.

Three parts. Each is designed to test something every lab so far has come back to - which error bar is the right one, whether the model needs the extra parameter, and whether the check you ran could have failed. You've now got the tools to do all of this, so let's do it on numbers you haven't seen: 250 real Type Ia supernovae, and someone else's analysis of real white dwarfs.

Part 1 is 30 pts (6 each), Part 2 is 45 pts, and Part 3 is 25 pts. You'll need 2b and 2c right to get 2d right.

Remember, the rules are the syllabus rules - open book, you can use AI tools (document them in 3b), you can work together and discuss problems and solutions, but you have to write up your own solution and run it on **your own data** in `data/<your netid>/` (nobody else's numbers will match yours).

Put your notebook in `submissions/<your netid>/midterm.ipynb`. The path below (`DATA`) reaches your files from there. Make sure it runs top to bottom from a fresh kernel ("Restart & Run All") - that is how it will be graded, and how it will be run if you defend it.

Throughout, make sure every number you quote has a unit, an error bar and a sensible number of digits (two significant figures on the error bar, and round the value to match).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

NETID = 'example'   # <-- an example data set made the same way as yours; use your own netid on the exam
DATA = f'data/{NETID}/'   # the exam used ../../labs/midterm/data/{NETID}/ from submissions/<your netid>/

def load(name):
    """Read one of your CSV files as a pandas table; d['column'].to_numpy() gives a numpy array. Names stay strings."""
    return pd.read_csv(DATA + name + '.csv', dtype={'name': str})

In [ ]:
# extra imports used by the solution below
import emcee, corner
import arviz as az
from astropy.cosmology import FlatLambdaCDM
COSMO = FlatLambdaCDM(H0=70, Om0=0.3)
rng = np.random.default_rng(457)

## Part 1: Error bars (30 pts, 6 each)

Five short scenarios. Each one is a number someone handed you - say whether it's right, compute the right one, and say in a sentence or two why. Show the code. Each of these should be about 20 minutes of work.

### 1a - The extinction coefficient (`p1a_extinction.csv`)

An observer measured a photometric zero point at 15 different airmasses through one night. Extinction in magnitudes grows in proportion to the airmass $X$, so the zero point should follow a line, $m = k\,X + m_0$, where $k$ is the extinction coefficient in mag per airmass. Columns: `airmass`, `zeropoint_mag`, `sigma_quoted_mag` (the pipeline's error on each point).

1. Fit the line by weighted least squares with the quoted errors. Report $k$ and its error bar computed from the quoted errors alone.
2. Now run `np.polyfit(airmass, zeropoint_mag, 1, w=1/sigma_quoted_mag, cov=True)` and report its error bar on $k$. Explain exactly what `polyfit` did to get a different number (read its documentation for `cov`).
3. Compute $\chi^2$ and the number of degrees of freedom. How probable is a $\chi^2$ this large or this small if the quoted errors were right? (Compute it from the $\chi^2$ distribution.)
4. Report the error bar on $k$ you'd quote, and justify it. What does your answer to (3) tell you about the pipeline's error bars?

In [ ]:
# 1a: your work here
a = load('p1a_extinction')
X, zp, sig = a['airmass'].to_numpy(), a['zeropoint_mag'].to_numpy(), a['sigma_quoted_mag'].to_numpy()

# (1) weighted least squares with the quoted errors alone
A = np.column_stack([X, np.ones_like(X)])
W = 1 / sig**2
cov = np.linalg.inv(A.T @ (A * W[:, None]))
k, m0 = cov @ (A.T @ (W * zp))
sig_k = np.sqrt(cov[0, 0])
print(f"(1) WLS, quoted errors only: k = {k:.4f} +/- {sig_k:.4f} mag/airmass;  m0 = {m0:.4f} +/- {np.sqrt(cov[1, 1]):.4f} mag")

# (2) polyfit with cov=True
p_pf, cov_pf = np.polyfit(X, zp, 1, w=1 / sig, cov=True)
sig_k_pf = np.sqrt(cov_pf[0, 0])
chi2 = np.sum(((zp - A @ [k, m0]) / sig)**2)
dof = len(X) - 2
print(f"(2) polyfit(cov=True): k = {p_pf[0]:.4f} +/- {sig_k_pf:.4f} mag/airmass")
print(f"    ratio polyfit/quoted-only = {sig_k_pf / sig_k:.3f};  sqrt(chi2/dof) = {np.sqrt(chi2 / dof):.3f}")

# (3) chi-square and its tail probabilities
p_low, p_high = stats.chi2.cdf(chi2, dof), stats.chi2.sf(chi2, dof)
print(f"(3) chi2 = {chi2:.1f} for {dof} dof (chi2/dof = {chi2 / dof:.2f});  P(a new chi2 <= {chi2:.1f}) = {p_low:.2e};  P(a new chi2 >= {chi2:.1f}) = {p_high:.2e}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].errorbar(X, zp, sig, fmt='o', ms=4); xx = np.array([X.min(), X.max()]); ax[0].plot(xx, k * xx + m0)
ax[0].set_xlabel('airmass $X$'); ax[0].set_ylabel('zero point (mag)')
ax[1].errorbar(X, (zp - A @ [k, m0]) / sig, 1, fmt='o', ms=4); ax[1].axhline(0, color='k', lw=0.5)
ax[1].set_xlabel('airmass $X$'); ax[1].set_ylabel('normalized residual'); plt.tight_layout(); plt.show()
# where does the observed chi2 sit? (the tail we read in (3))
xs_ = np.linspace(0, max(35, 1.3 * chi2), 400)
plt.figure(figsize=(5.2, 3.2)); plt.plot(xs_, stats.chi2.pdf(xs_, dof), 'k')
plt.fill_between(xs_[xs_ <= chi2], stats.chi2.pdf(xs_[xs_ <= chi2], dof), color='C0', alpha=0.4, label=f'P(new chi2 <= {chi2:.1f}) = {p_low:.1e}')
plt.fill_between(xs_[xs_ >= chi2], stats.chi2.pdf(xs_[xs_ >= chi2], dof), color='C3', alpha=0.4, label=f'P(new chi2 >= {chi2:.1f}) = {p_high:.1e}')
plt.axvline(chi2, color='k', lw=0.8); plt.xlabel(f'$\\chi^2$ ({dof} dof)'); plt.ylabel('probability density'); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

<div style="color:#1f6feb">

**ANSWER:**


1. Weighted least squares with the quoted errors alone gives $k = 0.1348 \pm 0.0136$ mag/airmass (the intercept is $25.228 \pm 0.024$ mag).
2. `np.polyfit(..., w=1/sigma, cov=True)` gives $\pm 0.0052$ mag/airmass. Notice that when you pass weights and ask for `cov=True`, `polyfit` multiplies the covariance matrix by $\chi^2/(N-2)$ before it hands it back, so its error bar no longer comes from the quoted errors, but rather describes the scatter the fit actually saw. Check: $0.0136 \times \sqrt{\chi^2/\mathrm{dof}} = 0.0136 \times 0.386 = 0.0052$.
3. $\chi^2 = 1.9$ for $15 - 2 = 13$ degrees of freedom, so $\chi^2/\mathrm{dof} = 0.15$. The probability of getting a $\chi^2$ this small or smaller is $1.9\times10^{-4}$ (`stats.chi2.cdf`, the blue area in the last figure). The upper tail (`stats.chi2.sf`, the red area) is about 1, so it's the wrong one to read when our $\chi^2$ is far below the number of degrees of freedom. If the quoted errors were right, we'd expect a fit this good about once in 5000 tries.
4. I'd quote $k = 0.1348 \pm 0.0052$ mag/airmass, the scatter-based error bar. The data scatter 2.6 times less than the quoted errors say they should, so I trust the scatter I can see over the errors the pipeline claims, and the pipeline's error bars are too big by about $1/0.386 = 2.6$. With 13 degrees of freedom a standard deviation measured from the scatter is only good to about $1/\sqrt{2\times13} = 20\%$, which can't explain a factor of 2.6. If the scatter had agreed with the quoted errors there would be nothing to rescale, and I'd quote the quoted-errors answer. Look at your own $\chi^2/\mathrm{dof}$ before you write this sentence - it's below 1 for some students' data and above 1 for others, and the answer flips with it.

</div>

**Points (6):** 1 for the weighted fit with the quoted errors alone. 2 for the `polyfit` error bar and the reason it is different (it rescales by $\chi^2/\mathrm{dof}$). 1.5 for $\chi^2$, the degrees of freedom and a p-value from the $\chi^2$ distribution, read in the tail that matches your $\chi^2$. 1.5 for the error bar you quote and a conclusion about the pipeline that agrees with your $\chi^2$.

### 1b - The centroid (`p1b_centroid.csv`)

A classmate measured the position of a star from `n_photons` photons, each landing at a position drawn from a Gaussian of width `psf_sigma_arcsec` around the true position. They report an error on the position of `quoted_sigma_x0_arcsec`, from a clever weighting scheme they found online.

1. Compute the Cramér-Rao bound on the error of any unbiased estimator of the position (Lab 02).
2. Is the quoted error possible? If not, say what the classmate's number must mean instead, and name two concrete mistakes that would produce an error bar this small.

In [ ]:
# 1b: your work here
b = load('p1b_centroid').iloc[0]
psf, N_ph, quoted = b['psf_sigma_arcsec'], b['n_photons'], b['quoted_sigma_x0_arcsec']
crb = psf / np.sqrt(N_ph)
n_needed = (psf / quoted)**2
print(f"PSF sigma = {psf:.3f} arcsec, N = {N_ph:.0f} photons")
print(f"Cramer-Rao bound = sigma_PSF/sqrt(N) = {crb:.4f} arcsec;  quoted = {quoted:.4f} arcsec;  quoted/CRB = {quoted / crb:.2f}")
print(f"The quoted error would need N = (sigma_PSF/quoted)^2 = {n_needed:.0f} photons, not {N_ph:.0f}")

<div style="color:#1f6feb">

**ANSWER:**


1. For $N$ photons drawn from a Gaussian of width $\sigma_{\rm PSF}$, the Cramér-Rao bound on the centroid is $\sigma_{x_0} \ge \sigma_{\rm PSF}/\sqrt{N} = 1.252/\sqrt{462} = 0.0582$ arcsec.
2. The quoted $0.0445$ arcsec is $0.76$ times the bound, so it's not possible for an unbiased estimator. To get $0.0445$ arcsec from this PSF you'd need $N = (\sigma_{\rm PSF}/0.0445)^2 = 792$ photons, $1.7$ times the 462 we have. It could be the error of a biased estimator (a weighting scheme that pulls the answer toward the position you expect can beat the bound, but then the answer is biased), or the error on a different quantity, such as an error per axis or a relative error.

For the two mistakes: (a) counting photons twice, for example adding two exposures that share photons, or counting sky photons as source photons, so $N$ looks too big. (b) Choosing the weights from the same photons (say, weighting each photon by how close it is to the current centroid) and then quoting the error of the weighted mean as if the weights were fixed. The weights follow the data's own noise, so the estimate hugs the sample and the scatter looks smaller than it really is.

</div>

**Points (6):** 2 for the Cramér-Rao bound. 1 for saying the quoted error is not possible. 1 for saying what the quoted number must be instead. 2 for two concrete mistakes that would give an error this small (1 each - "they made an error" is 0).

### 1c - The transit depth's error bar (`p1c_profile.csv`)

A transit-depth fit produced this profile: columns `depth_ppt` (the depth, in parts per thousand) and `minus2lnL` ($-2\ln L$ at that depth, other parameters fixed at their best values). A classmate took the second derivative of `minus2lnL` at its minimum and reported $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$.

1. Find the best depth and the correct $1\sigma$ error from the curvature. Is the classmate's error bar right? If not, by what factor is it off, and why?
2. Read the $\Delta(-2\ln L) = 1$ interval directly off the profile. Is it symmetric? Compare it with your curvature answer.
3. If the classmate quotes $\pm$ their $\sigma$ on many independent datasets, what fraction of those intervals contain the true depth? Predict it, then check it with a simulation (a few thousand fake best-fit depths drawn around a truth you choose, using your correct $\sigma$).

In [ ]:
# 1c: your work here
c1 = load('p1c_profile')
dep, m2 = c1['depth_ppt'].to_numpy(), c1['minus2lnL'].to_numpy()
near = m2 < m2.min() + 4                      # parabola over the region within about 2 sigma of the minimum
q = np.polyfit(dep[near], m2[near], 2)        # -2lnL ~ q0 d^2 + q1 d + q2
best = -q[1] / (2 * q[0]); curv = 2 * q[0]    # curv = d^2(-2lnL)/d depth^2
min_val = np.polyval(q, best)
sig_class = 1 / np.sqrt(curv)                 # the classmate's number
sig_right = np.sqrt(2 / curv)                 # Delta(-2lnL) = 1  <=>  (1/2) curv delta^2 = 1
print(f"best depth = {best:.3f} ppt;  curvature = {curv:.1f} ppt^-2")
print(f"classmate sigma = {sig_class:.3f} ppt;  correct sigma = {sig_right:.3f} ppt;  ratio = {sig_right / sig_class:.3f} (sqrt 2 = {np.sqrt(2):.3f})")

# (2) read the Delta(-2lnL) = 1 interval straight off the profile
i0 = np.argmin(m2)
lo = np.interp(min_val + 1, m2[:i0 + 1][::-1], dep[:i0 + 1][::-1])
hi = np.interp(min_val + 1, m2[i0:], dep[i0:])
print(f"Delta(-2lnL)=1 interval: [{lo:.3f}, {hi:.3f}] ppt = {best:.3f} -{best - lo:.3f} +{hi - best:.3f};  curvature sigma = {sig_right:.3f}")

# (3) coverage of +/- the classmate's sigma
pred = 2 * stats.norm.cdf(1 / np.sqrt(2)) - 1
truth = best; draws = np.random.default_rng(1).normal(truth, sig_right, 5000)
cov_sim = np.mean(np.abs(draws - truth) < sig_class)
print(f"predicted coverage 2*Phi(1/sqrt2) - 1 = {pred:.4f};  simulated (5000 draws) = {cov_sim:.4f} +/- {np.sqrt(cov_sim * (1 - cov_sim) / 5000):.4f}")

plt.figure(figsize=(5, 3.2)); plt.plot(dep, m2, '.'); plt.plot(dep, np.polyval(q, dep), lw=1)
plt.axhline(min_val + 1, color='C3', lw=0.7); plt.ylim(m2.min() - 1, m2.min() + 8)
plt.xlabel('depth (ppt)'); plt.ylabel('$-2\\ln L$'); plt.tight_layout(); plt.show()

<div style="color:#1f6feb">

**ANSWER:**


1. Let's expand $-2\ln L$ about its minimum. Near the minimum $-2\ln L \approx \min + \tfrac12\,{\rm curv}\,\delta^2$, where $\delta$ is the distance from the best depth and ${\rm curv} = d^2(-2\ln L)/d\,{\rm depth}^2$. Then $\Delta(-2\ln L) = 1$ happens at $\delta^2 = 2/{\rm curv}$, so $\sigma = \sqrt{2/{\rm curv}}$. The best depth is $3.67$ ppt (the vertex of a parabola fitted to $-2\ln L$ within $\Delta(-2\ln L) = 4$ of the minimum) and ${\rm curv} = 20.0$ ppt$^{-2}$, which gives $\sigma = 0.316$ ppt, so the depth is $3.67 \pm 0.32$ ppt. The classmate's $1/\sqrt{20.0} = 0.223$ ppt is too small by $\sqrt 2$. They used the curvature of $-2\ln L$ as if it were the curvature of $-\ln L$, which is half as big, and it's $1/\sqrt{{\rm curvature\ of\ }-\ln L}$ that gives the error bar.
2. Reading $\Delta(-2\ln L) = 1$ directly off the profile gives $[3.355, 3.986]$ ppt, which is $-0.316/+0.316$. For this file the profile is symmetric to better than 1%, so it agrees with the curvature answer of $0.316$. Other students' profiles are a little skewed (up to about 7%), so say what you see in yours.
3. The classmate's interval is $\pm 0.223 = \pm 0.707\,\sigma$ (that's $0.223/0.316$), and a Gaussian covers $\pm 0.707\,\sigma$ with probability $2\Phi(1/\sqrt 2) - 1 = 0.5205$. I simulated 5000 best-fit depths from a Gaussian around a truth I chose, with the correct $\sigma$, and counted how often $\pm 0.223$ contained the truth: $0.522 \pm 0.007$. So the classmate's "68%" interval contains the truth about half the time - and we called it 68%! (This simulation is just the formula evaluated by Monte Carlo, so it can't disagree with it. A test that could fail would simulate fake data sets, fit each one, and compute the classmate's $\sigma$ from the curvature of each fit.)

</div>

**Points (6):** 1.5 for the best depth and the curvature error. 1 for the $\sqrt2$ and the reason for it. 1.5 for the $\Delta(-2\ln L)=1$ interval and its comparison with the curvature answer. 2 for the predicted coverage and the simulation that checks it (1 each).

### 1d - The companion mass (`p1d_mass_samples.csv`)

These are 8000 posterior samples of the mass of an unseen companion, `M2_Msun`. Three classmates report its uncertainty as (i) the variance of the samples, (ii) the 84th percentile minus the 16th, and (iii) the standard deviation divided by $\sqrt{8000}$.

1. Report the mass as a median with its 16th-84th percentile error bars, in solar masses.
2. For each of (i)-(iii): what quantity is it actually, what are its units, and by roughly what factor is it wrong as a $1\sigma$ error bar?
3. Is a symmetric $\pm$ error bar appropriate for this posterior? If not, show why.

In [ ]:
# 1d: your work here
M2 = load('p1d_mass_samples')['M2_Msun'].to_numpy()
lo16, med, hi84 = np.percentile(M2, [16, 50, 84])
print(f"M2 = {med:.2f} +{hi84 - med:.2f} -{med - lo16:.2f} Msun   (N = {M2.size})")
var_i, width_ii, sem_iii = M2.var(ddof=1), hi84 - lo16, M2.std(ddof=1) / np.sqrt(M2.size)
half = 0.5 * width_ii
print(f"(i)   variance = {var_i:.2f} Msun^2   (a squared width; half-width = {half:.2f} Msun)")
print(f"(ii)  84th - 16th = {width_ii:.2f} Msun = {width_ii / half:.1f} x the half-width, i.e. about 2 sigma")
print(f"(iii) std/sqrt(N) = {sem_iii:.4f} Msun;  half-width / this = {half / sem_iii:.0f}  (sqrt(8000) = {np.sqrt(M2.size):.0f})")
print(f"skewness of M2 = {stats.skew(M2):.2f};  skewness of ln M2 = {stats.skew(np.log(M2)):.2f}")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].hist(M2, 60, color='0.7'); [ax[0].axvline(v, color='C3', lw=0.8) for v in (lo16, med, hi84)]; ax[0].set_xlabel('$M_2$ ($M_\\odot$)')
ax[1].hist(np.log(M2), 60, color='0.7'); ax[1].set_xlabel('$\\ln M_2$'); plt.tight_layout(); plt.show()

<div style="color:#1f6feb">

**ANSWER:**


1. $M_2 = 11.99\,^{+2.50}_{-2.11}\ M_\odot$ (median and 16th-84th percentiles of the 8000 samples).
2. (i) The variance is $5.46\ M_\odot^2$. Its units are $M_\odot^2$, so it's a squared width and can't be an error bar in $M_\odot$. Its square root, $2.34\ M_\odot$, is a standard deviation, and it's close to the half-width of the 68% interval, $(84{\rm th} - 16{\rm th})/2 = 2.31\ M_\odot$, but that's a coincidence of these numbers. There's no factor to quote for (i) because the units differ. (ii) The 84th minus the 16th percentile is $4.61\ M_\odot$, i.e. the full width of the central 68%, about $2\sigma$. In other words, as a $1\sigma$ error bar it's $2\times$ too big. (iii) $\sigma/\sqrt{8000} = 0.0261\ M_\odot$ is the Monte Carlo error on the mean of the samples. It says how well we know the average of the samples, and that's a different question from how well we know the mass. It's about $\sqrt{8000} = 89$ times (88 here) too small.
3. No. The error bars aren't symmetric ($+2.50$ against $-2.11$). The skewness of $M_2$ is $0.58$ but the skewness of $\ln M_2$ is $0.00$, so the posterior is close to lognormal, i.e. symmetric in $\ln M_2$ (see the histograms). I'd quote the asymmetric percentiles as in item 1, or quote $\ln M_2$ with a symmetric $\pm$.

</div>

**Points (6):** 1.5 for the median with its 16th-84th percentile error bars, in solar masses. 3 for (i), (ii) and (iii), 1 for each of them, which means saying what the quantity is, its units and how far off it is as a $1\sigma$ error bar. 1.5 for saying a symmetric $\pm$ is not appropriate and showing why.

### 1e - The slope at fixed intercept (`p1e_slope_intercept_samples.csv`)

Posterior samples of a slope and an intercept from a line fit. A classmate fixed the intercept at its best value, refit only the slope, and quoted the slope's error from that 1-D fit.

1. Compute the slope's error bar the classmate got (the spread of the slope *at fixed intercept*), and the marginal one (the spread of the slope with the intercept marginalized over).
2. Compute their ratio, and the correlation coefficient $\rho$ of the samples. How does the ratio depend on $\rho$? Plot the samples and draw both error bars on the plot.
3. When is the classmate's number the right one to quote? One sentence.

In [ ]:
# 1e: your work here
s = load('p1e_slope_intercept_samples'); sl, ic = s['slope'].to_numpy(), s['intercept'].to_numpy()
rho = np.corrcoef(sl, ic)[0, 1]
sig_marg = sl.std(ddof=1)
bfit = np.polyfit(ic, sl, 1)                       # slope as a function of intercept
sig_cond = (sl - np.polyval(bfit, ic)).std(ddof=2) # spread of the slope AT FIXED intercept
print(f"marginal sigma(slope) = {sig_marg:.4f};  fixed-intercept sigma(slope) = {sig_cond:.4f}")
print(f"ratio = {sig_cond / sig_marg:.3f};  rho = {rho:.3f};  sqrt(1 - rho^2) = {np.sqrt(1 - rho**2):.3f}")

plt.figure(figsize=(5.2, 4)); plt.plot(ic, sl, '.', ms=1.5, alpha=0.4, color='0.5')
x0, y0 = ic.mean(), sl.mean()
plt.errorbar([x0], [y0], yerr=sig_marg, fmt='o', color='C0', capsize=5, label=f'marginal $\\pm${sig_marg:.3f}')
plt.errorbar([x0 + 0.1 * ic.std()], [y0], yerr=sig_cond, fmt='o', color='C3', capsize=5, label=f'fixed intercept $\\pm${sig_cond:.3f}')
plt.xlabel('intercept'); plt.ylabel('slope'); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

<div style="color:#1f6feb">

**ANSWER:**


1. The marginal error is the spread of the slope samples with the intercept marginalized over: $\sigma_{\rm marg} = 0.0344$. The classmate's fixed-intercept error is the scatter left in the slope once I predict it from the intercept with a straight line: $\sigma_{\rm cond} = 0.0166$.
2. The ratio is $0.482$ and $\rho = -0.876$. Let's assume the slope and intercept are jointly Gaussian. Then the spread of one at a fixed value of the other is $\sigma\sqrt{1-\rho^2}$, which is exactly the $0.482$ we measure. The more strongly the two are correlated, the more the fixed-intercept error understates the uncertainty. This connects to the Fisher information: the classmate's number is $1/\sqrt{F_{ss}}$, where $F$ is the Fisher matrix, and the marginal error is $\sqrt{(F^{-1})_{ss}}$, the matching diagonal element of the inverse. In the plot the blue bar (marginal) is taller than the red one (fixed intercept); I shifted the red one sideways so they don't overlap.
3. The classmate's number is the right one to quote only when the intercept really is known (fixed by a calibration or an exact constraint), or when the slope and intercept are uncorrelated ($\rho \approx 0$).

</div>

**Points (6):** 1.5 for both error bars. 1.5 for the ratio and $\rho$, and for noticing that the ratio is $\sqrt{1-\rho^2}$. 1.5 for the plot with both error bars on the samples. 1.5 for when the classmate's number is the right one.

## Part 2: 250 supernovae and a step (45 pts)

Type Ia supernovae are standardizable candles: their peak brightness, corrected for how fast they fade and for their color, measures distance well enough to have discovered cosmic acceleration. Your file `p2_pantheon_sne.csv` holds 250 real SNe Ia from the Pantheon+ compilation (Scolnic et al. 2022, ApJ 938, 113; [Brout et al. 2022](https://arxiv.org/abs/2202.04077), ApJ 938, 110), all at redshift $0.023 < z < 0.15$. For each supernova, a light-curve fit (SALT2) gives:

- `mB`: the peak apparent magnitude in the B band, and `mB_err`;
- `x1`: the light-curve *stretch* (broader light curves are brighter), and `x1_err`;
- `c`: the *color* (redder supernovae are fainter, from dust and intrinsically), and `c_err`;
- `cov_mB_x1`, `cov_mB_c`, `cov_x1_c`: the covariances between those three, from the same light-curve fit;
- `z`: the redshift; `host_logmass`, `host_logmass_err`: $\log_{10}$ of the host galaxy's stellar mass in solar masses, and its error.

The standardized brightness is the **Tripp relation**,

$$m_B = M + \mu(z) - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10),$$

where $M$ is the absolute magnitude of a "standard" SN Ia, $\mu(z)$ the distance modulus, $\Theta$ is 1 for hosts more massive than $10^{10}\,M_\odot$ and 0 otherwise, and $\gamma$ is the **host-mass step**. Supernovae in massive hosts appear to be brighter after standardization (Kelly et al. 2010; Sullivan et al. 2010); whether the step is real, and why, matters for supernova dark-energy measurements. In this form, a step that makes supernovae in massive hosts brighter has $\gamma < 0$; the published value is about $\gamma \approx -0.06$ mag. So the question for your sample is whether you need $\gamma$ at all.

There are two more things you'll need. At these redshifts the galaxies' own motions (peculiar velocities, about 250 km/s) shift $z$, and so $\mu$; add $\sigma_{\mu,\rm pec} = \frac{5}{\ln 10}\frac{250\ {\rm km/s}}{c\,z}$ to each supernova's variance. And no supernova is a perfect standard candle: add an intrinsic scatter $\sigma_{\rm int}$, a free parameter. Use a flat $\Lambda$CDM distance modulus with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$ (`astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z)`); $M$ absorbs the choice of $H_0$.

We planted between **two and four contaminants** in your file - objects whose `mB` we made fainter by more than a magnitude, so they are not standard SNe Ia any more (think of a mis-classified peculiar supernova). Their names are not in the file. Real data have a few real outliers too, so not everything far from the line is planted.

### 2a - Look at the data (5 pts)

1. Compute $\mu(z)$ and plot the **Hubble residuals** $m_B - \mu(z)$ against $x_1$, against $c$, and against `host_logmass`.
2. Which way do the $x_1$ and $c$ trends go? Is that the sign the Tripp relation expects? Two or three sentences.
3. The raw trend with host mass may not have the sign of $\gamma$. Why not? (Plot $x_1$ against host mass.)
4. Can you see any of the contaminants?
5. What units do $\alpha$ and $\beta$ have?

In [ ]:
# 2a: your work here
sn = load('p2_pantheon_sne')
z, mB, x1, col, lm = (sn[k].to_numpy() for k in ('z', 'mB', 'x1', 'c', 'host_logmass'))
mu = COSMO.distmod(z).value
hres = mB - mu                                   # Hubble residual (absorbs M)

def binned(x, y, n=8):
    edges = np.quantile(x, np.linspace(0, 1, n + 1)); idx = np.clip(np.digitize(x, edges) - 1, 0, n - 1)
    return np.array([x[idx == i].mean() for i in range(n)]), np.array([y[idx == i].mean() for i in range(n)])

fig, ax = plt.subplots(1, 4, figsize=(15, 3.3))
for a_, (x, lab) in zip(ax[:3], [(x1, '$x_1$'), (col, '$c$'), (lm, '$\\log_{10} M_\\star/M_\\odot$')]):
    a_.plot(x, hres, '.', ms=3, color='0.6'); a_.plot(*binned(x, hres), 'o-', color='C3'); a_.set_xlabel(lab); a_.set_ylabel('$m_B - \\mu(z)$ (mag)')
ax[3].plot(lm, x1, '.', ms=3, color='0.6'); ax[3].plot(*binned(lm, x1), 'o-', color='C3'); ax[3].set_xlabel('$\\log_{10} M_\\star/M_\\odot$'); ax[3].set_ylabel('$x_1$')
for a_ in ax[:3]: a_.axvline(10, color='k', lw=0.4) if a_ is ax[2] else None
plt.tight_layout(); plt.show()
print(f"slope of residual vs x1:    {np.polyfit(x1, hres, 1)[0]:+.3f} mag per unit x1")
print(f"slope of residual vs c:     {np.polyfit(col, hres, 1)[0]:+.2f} mag per unit c")
print(f"slope of residual vs mass:  {np.polyfit(lm, hres, 1)[0]:+.3f} mag per dex")
print(f"corr(x1, mass) = {np.corrcoef(x1, lm)[0, 1]:+.2f};  mean x1: low-mass hosts {x1[lm < 10].mean():+.2f}, high-mass hosts {x1[lm > 10].mean():+.2f}")

# contaminants: regress out x1 and c only, look at the biggest positive (fainter) residuals
Aq = np.column_stack([np.ones_like(x1), x1, col]); coef, *_ = np.linalg.lstsq(Aq, hres, rcond=None)
rr = hres - Aq @ coef
print("largest (fainter) residuals after a plain x1,c regression:", [(sn['name'][i], round(rr[i] / rr.std(), 1)) for i in np.argsort(rr)[::-1][:6]])
# the same residual against host mass, raw and after removing the x1 and c terms (plain regression)
std_res = hres - (Aq[:, 1:] @ coef[1:])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.3), sharey=True)
for a_, y_, lab in zip(ax, (hres, std_res), ('raw $m_B - \\mu$', 'after removing $x_1$ and $c$ terms')):
    a_.plot(lm, y_, '.', ms=3, color='0.6'); a_.plot(*binned(lm, y_), 'o-', color='C3'); a_.axvline(10, color='k', lw=0.4)
    a_.set_xlabel('$\\log_{10} M_\\star/M_\\odot$'); a_.set_title(lab, fontsize=9)
ax[0].set_ylabel('mag'); plt.tight_layout(); plt.show()
print(f"mean residual, low-mass minus high-mass hosts: raw {hres[lm < 10].mean() - hres[lm > 10].mean():+.3f} mag, after x1,c terms {std_res[lm < 10].mean() - std_res[lm > 10].mean():+.3f} mag")

<div style="color:#1f6feb">

**ANSWER:**


1. The plots show the Hubble residual $m_B - \mu(z)$ against $x_1$, $c$ and host mass (red points are binned means), and $x_1$ against host mass.
2. The residual falls with $x_1$ (slope $-0.17$ mag per unit $x_1$), so broader supernovae are brighter. It rises with $c$ (slope $+2.7$ mag per unit $c$), so redder supernovae are fainter. Those are the signs the Tripp relation expects ($-\alpha x_1 + \beta c$ with $\alpha, \beta > 0$).
3. A larger magnitude means fainter, so keep that in mind for the signs. The raw residual rises with host mass ($+0.11$ mag/dex), the opposite of what $\gamma < 0$ (massive hosts brighter) would give. The reason is that $x_1$ and host mass are anticorrelated ($r = -0.37$ - the mean $x_1$ is $+0.16$ in low-mass hosts and $-0.48$ in high-mass hosts). Massive hosts have narrower light curves (low $x_1$), and those are fainter, so the stretch effect dominates the raw trend. If we remove the fitted $x_1$ and $c$ terms the raw trend goes away (last figure): low-mass hosts go from $0.14$ mag brighter than high-mass hosts in the raw residuals to $0.016$ mag fainter. In other words, the step has to be fitted together with $\alpha$ and $\beta$.
4. None of the contaminants is obvious in the first figure. The highest points sit at low $x_1$ and high $c$, where the scatter is large anyway. After a plain regression on $x_1$ and $c$, four supernovae are $5.3$-$6.1\sigma$ outliers and the next one is at $2.3\sigma$ (see the printout), and in the last figure they're the points well above the rest. So you can see them once the trends are out.
5. $\alpha$ and $\beta$ are in magnitudes. $x_1$ and $c$ are dimensionless (a stretch and a color difference), and $\alpha x_1$ and $\beta c$ have to be magnitudes to add to $m_B$, which is what "mag per unit $x_1$" and "mag per unit $c$" mean. Some people call them dimensionless because a magnitude is a log ratio. I'd write mag.

</div>

**Points (5):** 1 for the three residual plots. 1 for the signs of the $x_1$ and $c$ trends and whether they are what Tripp expects. 1.5 for why the raw host-mass trend need not have the sign of $\gamma$, backed by the $x_1$-mass plot. 1 for what you can and can't see of the contaminants. 0.5 for the units of $\alpha$ and $\beta$.

### 2b - The likelihood (10 pts)

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$. Each supernova's variance is

$$\sigma_i^2 = \mathbf{v}^T C_i\,\mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2, \qquad \mathbf{v} = (1, \alpha, -\beta),$$

where $C_i$ is that supernova's $3\times3$ covariance of $(m_B, x_1, c)$.

1. Write out $\mathbf{v}^T C_i\,\mathbf{v}$ term by term, and explain in one sentence why the off-diagonal terms are there.
2. Write the log-likelihood. Remember the $\ln\sigma_i^2$ term - the variance depends on the parameters.
3. State a prior for every parameter, and justify each in one line.
4. Decide whether to sample $\ln\sigma_{\rm int}$ or $\sigma_{\rm int}$, and say why.
5. Decide what to do about the contaminants. You may model them or remove them, but say how you found them and what a cut would cost you.

In [ ]:
# 2b: your work here
dat = {k: sn[k].to_numpy() for k in sn.columns if k != 'name'}
dat['mu'] = mu
dat['var_pec'] = (5 / np.log(10) * 250 / (2.998e5 * dat['z']))**2
dat['step'] = (dat['host_logmass'] > 10).astype(float)
names = ['M', 'alpha', 'beta', 'gamma', 'ln_sigint']
LO = np.array([-21., 0., 0., -0.2, np.log(0.01)]); HI = np.array([-17., 0.5, 6., 0.2, 0.])   # prior ranges

def subset(sel): return {k: v[sel] for k, v in dat.items()}

def var_i(theta, D):
    M, a, b, g, ls = theta
    # v^T C v with v = (1, alpha, -beta), term by term
    vCv = D['mB_err']**2 + a**2 * D['x1_err']**2 + b**2 * D['c_err']**2 + 2 * a * D['cov_mB_x1'] - 2 * b * D['cov_mB_c'] - 2 * a * b * D['cov_x1_c']
    return vCv + D['var_pec'] + np.exp(2 * ls)

def lnlike(theta, D):
    M, a, b, g, ls = theta
    r = D['mB'] - (M + D['mu'] - a * D['x1'] + b * D['c'] + g * D['step'])
    v = var_i(theta, D)
    return -0.5 * np.sum(r**2 / v + np.log(v) + np.log(2 * np.pi))

def lnpost(theta, D):
    theta = np.asarray(theta)
    return -np.inf if np.any(theta <= LO) or np.any(theta >= HI) else lnlike(theta, D)

def std_resid(theta, D):
    M, a, b, g, ls = theta
    r = D['mB'] - (M + D['mu'] - a * D['x1'] + b * D['c'] + g * D['step'])
    return r / np.sqrt(var_i(theta, D))

def hessian(f, p, h=1e-4):
    n = len(p); H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            ei = np.zeros(n); ej = np.zeros(n); ei[i] = h; ej[j] = h
            H[i, j] = (f(p + ei + ej) - f(p + ei - ej) - f(p - ei + ej) + f(p - ei - ej)) / (4 * h * h)
    return H

def fit(D, start=(-19.3, 0.14, 2.9, -0.04, np.log(0.11))):
    return minimize(lambda p: -lnpost(p, D), start, method='Nelder-Mead', options=dict(maxiter=40000, xatol=1e-9, fatol=1e-9))

# contaminants: iterate fit -> standardized residuals s_i -> cut one-sided (fainter) at 5 sigma, until nothing new is cut
cut = np.zeros(len(sn), bool)
for it in range(5):
    r = fit(subset(~cut)); sr = std_resid(r.x, dat)           # standardized residual s_i of every SN under the fit to the current clean set
    if it == 0: sr0 = sr.copy()
    new = (sr > 5) & ~cut
    print(f"pass {it}: N used = {(~cut).sum()}, sigma_int = {np.exp(r.x[4]):.3f}; new cuts (s > 5): {[(sn['name'][i], round(sr[i], 1)) for i in np.where(new)[0]]}")
    if not new.any(): break
    cut |= new
good = ~cut
print(f"removed {cut.sum()} SNe: {list(sn['name'][cut])};  largest remaining |s| = {np.abs(sr[good]).max():.1f}")
Dg = subset(good)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.3), sharey=True)
for a_, s_, lab in zip(ax, (sr0, std_resid(fit(Dg).x, dat)), ('pass 0: fit to all 250', 'fit to the clean set')):
    a_.plot(dat['z'], s_, '.', ms=4, color='0.5'); a_.plot(dat['z'][cut], s_[cut], 'o', color='C3', ms=6, label='cut')
    a_.axhline(5, color='C3', lw=0.7, ls='--'); a_.axhline(-5, color='C3', lw=0.7, ls='--'); a_.set_xlabel('redshift'); a_.set_title(lab, fontsize=9)
ax[0].set_ylabel('standardized residual $s_i$'); ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# what a cut costs: refit with no cut, and with one-sided cuts at 3, 4 and 5 sigma (s_i from the clean fit above)
sr_clean = std_resid(fit(Dg).x, dat)
for thr in (None, 3.0, 4.0, 5.0):
    sel = np.ones(len(sn), bool) if thr is None else (sr_clean < thr)
    rr_ = fit(subset(sel))
    print(f"cut at s > {thr}:  N = {sel.sum()},  gamma = {rr_.x[3]:+.3f},  sigma_int = {np.exp(rr_.x[4]):.3f},  alpha = {rr_.x[1]:.3f},  beta = {rr_.x[2]:.2f}")

<div style="color:#1f6feb">

**ANSWER:**


1. With $\mathbf v = (1, \alpha, -\beta)$,
$$\mathbf v^T C_i \mathbf v = C_{mm} + \alpha^2 C_{x_1x_1} + \beta^2 C_{cc} + 2\alpha C_{m x_1} - 2\beta C_{m c} - 2\alpha\beta\,C_{x_1 c},$$
where $C_{mm} = {\tt mB\_err}^2$ and so on. The off-diagonal terms are there because the three SALT2 numbers come from fitting the same light curve, so their errors are correlated.
2. $\ln L = -\tfrac12\sum_i\left[\dfrac{(m_{B,i} - M - \mu_i + \alpha x_{1,i} - \beta c_i - \gamma\Theta_i)^2}{\sigma_i^2} + \ln\sigma_i^2 + \ln 2\pi\right]$, with $\sigma_i^2 = \mathbf v^T C_i \mathbf v + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2$. We need the $\ln\sigma_i^2$ term because $\sigma_i$ depends on $\alpha$, $\beta$ and $\sigma_{\rm int}$. Without it the fit can make the $\chi^2$ smaller just by making $\sigma_{\rm int}$ bigger.
3. All priors are flat in the variable I name. $M \sim U(-21, -17)$ (we expect about $-19.3$, so $\pm2$ mag is wide). $\alpha \sim U(0, 0.5)$ and $\beta \sim U(0, 6)$ (positive from the physics, and well above the usual $0.14$ and $3$). $\gamma \sim U(-0.2, 0.2)$ mag (symmetric about zero and three times the published $|0.06|$ - this width matters in 2d, where it sets how much an extra parameter is penalized). $\ln\sigma_{\rm int} \sim U(\ln 0.01, \ln 1)$ (no preferred scale - $0.01$ mag is below what 250 supernovae can resolve and $1$ mag is far above the visible scatter).
4. I sample $\ln\sigma_{\rm int}$. It's a positive scale parameter, so the log removes the boundary at zero and gives a prior with no preferred scale. As a bonus the posterior comes out close to Gaussian in $\ln\sigma_{\rm int}$ (2c shows it).
5. For the contaminants I fit all the supernovae, standardize each residual by its $\sigma_i$ (call the result $s_i$; it isn't the redshift), and cut the ones that are fainter than the relation by more than $5\sigma$, which is the direction contaminants go. Then I refit, recompute $s_i$ for all supernovae under the new fit, and repeat until nothing new is cut. In pass 0 (all 250) $\sigma_{\rm int} = 0.207$ and four supernovae are cut at $5.2$-$6.4\sigma$ (15171, 20625, AT2016hns, SN2017ms). Pass 1 cuts nothing, and the largest remaining $|s|$ is $3.7$. The problem says to expect two to four planted contaminants, and four is the top of that range. The printed table shows what the cut does. With no cut $\sigma_{\rm int} = 0.207$ ($83\%$ larger), $\beta = 2.48$ ($2.8\sigma$ low, using the $0.108$ error from 2c) and $\gamma = -0.027$ ($0.6\sigma$ from the clean value, using $0.0193$), while $\alpha$ hardly moves. A cut at $s > 3$ removes two good supernovae as well (244 left), and $\sigma_{\rm int}$ drops from $0.113$ to $0.104$: cutting on the data trims the tails and biases $\sigma_{\rm int}$ low, so I'd rather cut far from where the actual scatter is. Cuts at 4 and 5 give identical results. With a $5\sigma$ cut I expect about $0.0001$ false cuts among 250 clean Gaussian supernovae, and some outliers are genuine (the largest clean $|s|$ is $3.7$), so I keep them. A mixture (outlier) model would avoid the hard cut, but it has extra parameters.

</div>

**Points (10):** 2 for $\mathbf v^T C_i\mathbf v$ written out and why the off-diagonal terms are there. 3 for the log-likelihood, including the $\ln\sigma_i^2$ term. 2 for a stated, defended prior on each parameter (0.4 each). 1 for $\ln\sigma_{\rm int}$ against $\sigma_{\rm int}$. 2 for how you found the contaminants and what a cut costs.

### 2c - Sampling and convergence (12 pts)

1. Find the maximum a posteriori point. Then run **four independent chains** from starting points spread wider than the posterior (you choose how, and say how you chose). Any sampler is fine - your own Metropolis, `emcee` (one ensemble per chain), or `PyMC`.
2. Report $\hat R$ and the effective sample size for every parameter (`arviz.rhat`, `arviz.ess`, or `arviz.summary`), and say what values would have made you run longer.
3. Make a corner plot, and quote $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ as medians with 16th-84th percentile error bars.
4. Which parameters are correlated strongly? Comment on why. Then fix $M$ at its MAP value, refit $\gamma$, and compare that $\sigma_\gamma$ with the marginal one (cf. 1e).

`ArviZ` 1.x (install it with `conda install -c conda-forge pymc arviz`, as in `INSTALL.txt`) takes your chains as a dictionary of arrays shaped (chain, draw): `idata = az.from_dict({'posterior': {'alpha': a, 'beta': b}})`, then `az.summary(idata)`.

In [ ]:
# 2c: your work here
map_ = fit(Dg); th_map = map_.x
Hm = hessian(lambda p: -lnlike(p, Dg), th_map); Cm = np.linalg.inv(Hm); err_h = np.sqrt(np.diag(Cm))
print("MAP:", {n: round(float(v), 4) for n, v in zip(names, th_map)}, " sigma_int =", round(float(np.exp(th_map[4])), 4))
print("Hessian errors:", dict(zip(names, np.round(err_h, 4))))

# four chains: own Metropolis, proposal = Hessian covariance scaled by 2.38^2/d, starts drawn at 4 x the Hessian error from the MAP
def metropolis(start, nstep, D, rng_):
    L = np.linalg.cholesky(Cm * 2.38**2 / 5); th = np.array(start, float); lp = lnpost(th, D)
    out = np.empty((nstep, 5)); acc = 0
    for i in range(nstep):
        prop = th + L @ rng_.standard_normal(5); lpp = lnpost(prop, D)
        if np.log(rng_.uniform()) < lpp - lp: th, lp, acc = prop, lpp, acc + 1
        out[i] = th
    return out, acc / nstep

starts = []
while len(starts) < 4:
    s0 = th_map + 4 * err_h * rng.standard_normal(5)
    if np.isfinite(lnpost(s0, Dg)): starts.append(s0)
runs = [metropolis(s0, 30000, Dg, np.random.default_rng(100 + i)) for i, s0 in enumerate(starts)]
chains = np.array([r_[0][5000:] for r_ in runs])                      # (chain, draw, param); first 5000 steps = burn-in
print("acceptance:", [round(r_[1], 2) for r_ in runs])
idata = az.from_dict({'posterior': {n: chains[:, :, i] for i, n in enumerate(names)}})
summ = az.summary(idata); print(summ[['mean', 'sd', 'ess_bulk', 'ess_tail', 'r_hat']])

In [ ]:
flat = chains.reshape(-1, 5).copy(); flat[:, 4] = np.exp(flat[:, 4])             # sigma_int in mag
labels = ['M', r'$\alpha$', r'$\beta$', r'$\gamma$ (mag)', r'$\sigma_{\rm int}$ (mag)']
corner.corner(flat, labels=labels, quantiles=[0.16, 0.5, 0.84], show_titles=True, title_fmt='.3f'); plt.show()
q_ = np.percentile(flat, [16, 50, 84], axis=0)
for i, n in enumerate(['M', 'alpha', 'beta', 'gamma', 'sigma_int']):
    print(f"{n:10s} = {q_[1, i]:.4f} +{q_[2, i] - q_[1, i]:.4f} -{q_[1, i] - q_[0, i]:.4f}")
print("\ncorrelation matrix (M, alpha, beta, gamma, sigma_int):"); print(np.round(np.corrcoef(flat.T), 2))
print(f"mean x1 = {Dg['x1'].mean():+.3f}, mean c = {Dg['c'].mean():+.4f} (clean sample);  fraction of hosts above log M = 10: {Dg['step'].mean():.2f}")

# conditional vs marginal sigma_gamma: fix M at its MAP value, refit the other four
fixM = lambda p: -lnpost(np.r_[th_map[0], p], Dg)
rM = minimize(fixM, th_map[1:], method='Nelder-Mead', options=dict(maxiter=40000, xatol=1e-9, fatol=1e-9))
sg_cond = np.sqrt(np.linalg.inv(hessian(fixM, rM.x))[2, 2])
print(f"\nsigma_gamma, M fixed: {sg_cond:.4f} mag;  marginal: {err_h[3]:.4f} (Hessian), {flat[:, 3].std():.4f} (chains);  ratio {sg_cond / flat[:, 3].std():.2f}")
print(f"sigma_M: marginal {err_h[0]:.4f}; with gamma fixed at 0 it would be {np.sqrt(np.linalg.inv(hessian(lambda p: -lnlike(np.r_[p[:3], 0., p[3]], Dg), np.r_[th_map[:3], th_map[4]]))[0, 0]):.4f}")
fig, ax = plt.subplots(5, 1, figsize=(9, 8), sharex=True)
for i_, n_ in enumerate(names):
    for ch_ in chains[:, :, i_]: ax[i_].plot(ch_[::20], lw=0.4)
    ax[i_].set_ylabel(n_)
ax[-1].set_xlabel('step / 20 (after burn-in)'); plt.tight_layout(); plt.show()
gg = np.linspace(flat[:, 3].min(), flat[:, 3].max(), 300)
plt.figure(figsize=(5.5, 3.2)); plt.hist(flat[:, 3], 60, density=True, color='0.8', label='marginal (chains)')
plt.plot(gg, stats.norm.pdf(gg, rM.x[2], sg_cond), 'C3', label=f'$M$ fixed at MAP: $\\sigma_\\gamma$ = {sg_cond:.4f}')
plt.xlabel('$\\gamma$ (mag)'); plt.ylabel('posterior density'); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

<div style="color:#1f6feb">

**ANSWER:**


1. The MAP is $M = -19.284$, $\alpha = 0.1467$, $\beta = 2.777$, $\gamma = -0.0387$, $\sigma_{\rm int} = 0.1132$. I ran four chains of my own Metropolis sampler. The proposal is the Hessian covariance times $2.38^2/5$, each chain has 30,000 steps with the first 5,000 thrown away, and the acceptance is $0.29$. Each chain starts at the MAP plus 4 times the Hessian error times a standard normal in every coordinate (redrawn if it falls outside the prior), which puts the starts well outside the posterior.
2. $\hat R = 1.00$ for all five parameters, and the bulk effective sample size is $5400$-$6100$ (tail ESS $\ge 8000$). I would have run longer if $\hat R > 1.01$ or the bulk ESS was below 400 (about 100 per chain).
3. (The corner plot also shows the correlations, and in the trace plots the four chains overlap after burn-in.) $\alpha = 0.147 \pm 0.009$, $\beta = 2.78 \pm 0.11$, $\gamma = -0.039 \pm 0.019$ mag, $\sigma_{\rm int} = 0.114\,^{+0.009}_{-0.008}$ mag, and $M = -19.284 \pm 0.014$ mag. These agree with the MAP and the Hessian errors to a few percent, so the posterior is close to Gaussian.
4. The strongest correlation is between $M$ and $\gamma$ ($\rho = -0.73$). Raising $M$ makes every supernova fainter, and lowering $\gamma$ makes the high-mass half brighter again, so the two partly cancel - they're degenerate, because the step applies to about half the sample. Next comes $\alpha$-$\gamma$ ($-0.30$), because $x_1$ is anticorrelated with host mass. $M$-$\alpha$ ($+0.11$) and $M$-$\beta$ ($+0.13$) are weak, and so is everything involving $\beta$ ($|\rho| \le 0.27$, the largest with $\sigma_{\rm int}$). The mean color is $-0.0008$ (printed above), so changing $\beta$ doesn't change the average brightness and $M$ doesn't trade off against it. If I fix $M$ at its MAP value, $\sigma_\gamma = 0.0130$ mag against $0.0193$ marginal, a ratio of $0.68$. That's what 1e predicts: $\sqrt{1 - (-0.73)^2} = 0.68$. For $\gamma$ I'd quote the marginal error, $0.0193$, because we don't know $M$. It works the other way too: fixing $\gamma = 0$ shrinks $\sigma_M$ from $0.0136$ to $0.0094$.

</div>

**Points (12):** 2 for the MAP and four chains started well outside the posterior, with a stated way of choosing the starts. 3 for $\hat R$ and the effective sample size for every parameter, and a rule for when you would run longer. 2 for a readable corner plot. 3 for $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ with error bars (0.75 each). 2 for the strongest correlations with reasons, and the conditional against the marginal $\sigma_\gamma$.

### 2d - Model comparison (10 pts)

Compare $M_0$ ($\gamma = 0$) with $M_1$ ($\gamma$ free).

1. Compute $2\ln B_{1,0}$ with the Laplace approximation (Day 13), with a prior on $\gamma$ you state and defend. How does it change if that prior is twice as wide?
2. Compute $\textrm{BIC}_0 - \textrm{BIC}_1$ for comparison.
3. How strong is your answer on Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong, above 10 very strong)? Compare $\gamma$ with the published $-0.06$ mag. Could your 250 supernovae have detected a step that size? (One way: your $\sigma_\gamma$ tells you how many $\sigma$ a $-0.06$ step would be.)

In [ ]:
# 2d: your work here
n_used = good.sum()
lnL1 = lnlike(th_map, Dg)
nll0 = lambda p: -lnlike(np.r_[p[:3], 0., p[3]], Dg)
r0 = minimize(nll0, np.r_[th_map[:3], th_map[4]], method='Nelder-Mead', options=dict(maxiter=40000, xatol=1e-9, fatol=1e-9))
lnL0 = -r0.fun; H0 = hessian(nll0, r0.x)
H1 = Hm
def two_lnB(width):
    lnZ1 = lnL1 + 0.5 * 5 * np.log(2 * np.pi) - 0.5 * np.linalg.slogdet(H1)[1] + np.log(1 / width)
    lnZ0 = lnL0 + 0.5 * 4 * np.log(2 * np.pi) - 0.5 * np.linalg.slogdet(H0)[1]
    return 2 * (lnZ1 - lnZ0)
B1, B2 = two_lnB(0.4), two_lnB(0.8)
dBIC = 2 * (lnL1 - lnL0) - np.log(n_used)
print(f"max ln L: M1 {lnL1:.2f}, M0 {lnL0:.2f};  2 ln(Lmax ratio) = {2 * (lnL1 - lnL0):.2f};  Occam term = 2lnB - 2 ln(Lmax ratio) = {B1 - 2 * (lnL1 - lnL0):+.2f}")
print(f"2 ln B_10 (gamma ~ U[-0.2, 0.2]) = {B1:+.2f};  prior twice as wide (U[-0.4, 0.4]) = {B2:+.2f}  (change {B2 - B1:+.2f}; -2 ln 2 = {-2 * np.log(2):.2f})")
print(f"BIC_0 - BIC_1 = {dBIC:+.2f}  (positive favors the step)")
sg = np.sqrt(np.linalg.inv(H1)[3, 3])
print(f"gamma = {th_map[3]:+.3f} +/- {sg:.3f} mag;  a step of -0.06 mag is {abs(-0.06) / sg:.1f} sigma;  gamma is {abs(th_map[3] + 0.06) / sg:.1f} sigma from -0.06")

<div style="color:#1f6feb">

**ANSWER:**


1. If we approximate the posterior near the peak as a Gaussian, then $Z \approx L_{\max}\,(2\pi)^{k/2}\,|H|^{-1/2}\,\pi(\gamma)$, where $H$ is the Hessian of $-\ln L$ at the MAP and $k$ is the number of parameters. The priors on the shared parameters ($M$, $\alpha$, $\beta$, $\sigma_{\rm int}$) cancel in the ratio. I take $\gamma \sim U(-0.2, 0.2)$ mag, i.e. a density of $1/0.4$. That's three times the published $|0.06|$, so a step of that size, if it's there, is well inside it. The result is $2\ln B_{1,0} = -0.28$. It has two pieces: the likelihood gain is $2\Delta\ln L_{\max} = 3.95$, and the Occam term (printed above) is $-4.23$, so they nearly cancel. With a prior twice as wide, $U(-0.4, 0.4)$, the Occam term gets $2\ln 2 = 1.39$ more negative and $2\ln B = -1.66$. That's because the Occam term scales with the width of the prior when the posterior is much narrower than the prior.
2. $\mathrm{BIC}_0 - \mathrm{BIC}_1 = 2\Delta\ln L_{\max} - \ln n = 3.95 - \ln 246 = -1.55$ (positive would favor the step). It agrees with the Laplace result in sign and size.
3. The interesting thing here is a tension. $\gamma = -0.039 \pm 0.019$ mag is $2.0\sigma$ from zero, yet $2\ln B \approx 0$. A $2\sigma$ improvement in the likelihood isn't enough to overcome the Occam penalty for an extra parameter with a prior this wide. Both numbers are under 2 in size, which is weak on Day 8's scale, i.e. the data give no clear evidence either way. $\gamma$ is $1.1\sigma$ from the published $-0.06$ mag, so it's consistent with both zero and the published step. A step of $-0.06$ mag would be $0.06/0.0193 = 3.1\sigma$ in this sample, so we could detect it at about $3\sigma$, which is marginal. I'd call the data inconclusive: they leave the published step possible.

</div>

**Points (10):** 4 for $2\ln B_{1,0}$ from the Laplace approximation with a prior you state and defend (1 of the 4 is for what doubling the prior width does). 2 for $\mathrm{BIC}_0 - \mathrm{BIC}_1$ with the sign the right way round. 2 for a conclusion on Day 8's scale that matches your own number. 2 for comparing $\gamma$ with $-0.06$ mag and saying whether 250 supernovae could have detected a step that size.

### 2e - Testing your pipeline (8 pts)

Design one test of your pipeline that would come out differently if your pipeline were wrong, and run it. State **before** you run it what result would count as a failure. One suggestion: generate fake supernovae at your redshifts, stretches and colors from parameters you choose (including a $\gamma$ you choose), fit 30 or more such datasets (maximum-likelihood fits with curvature error bars are fine here; you don't need MCMC for each), and check how often your 68% intervals contain the truth.

<div style="color:#1f6feb">

**2e - failure criterion, written before I ran the simulation.**

The test: I draw 200 fake data sets that reuse my own redshifts, stretches, colors, covariances and host masses. The $x_1$ and $c$ values in my file play the role of the true values for the fake supernovae. The true parameters are $M=-19.30$, $\alpha=0.14$, $\beta=2.9$, $\gamma=-0.06$ mag and $\sigma_{\rm int}=0.11$ mag. I add noise to $(m_B, x_1, c)$ drawn from each supernova's own covariance matrix $C_i$, and I add $\sigma_{\rm int}$ and the peculiar-velocity scatter to $m_B$. Then I fit each one with the same MAP pipeline.

My pipeline fails if either of these holds:
1. the 68% interval from the Hessian covers the truth for any of the five parameters less than 0.60 or more than 0.76 of the time (68% $\pm$ 2 binomial sigma for 200 sets is $\pm 0.066$);
2. the mean pull $(\hat\gamma - \gamma_{\rm true})/\sigma_\gamma$ is more than 3 standard errors from zero (about 0.21 for 200 sets), and the same for $\alpha$, $\beta$ and $\ln\sigma_{\rm int}$.

</div>

In [ ]:
# 2e: your work here
Cfull = np.zeros((len(sn), 3, 3))
Cfull[:, 0, 0], Cfull[:, 1, 1], Cfull[:, 2, 2] = dat['mB_err']**2, dat['x1_err']**2, dat['c_err']**2
Cfull[:, 0, 1] = Cfull[:, 1, 0] = dat['cov_mB_x1']; Cfull[:, 0, 2] = Cfull[:, 2, 0] = dat['cov_mB_c']; Cfull[:, 1, 2] = Cfull[:, 2, 1] = dat['cov_x1_c']
Lc = np.linalg.cholesky(Cfull)                       # all covariances are positive definite in this file
truth_ = np.array([-19.30, 0.14, 2.9, -0.06, np.log(0.11)])
sim_rng = np.random.default_rng(2026)
est, err = [], []
for isim in range(200):
    e3 = np.einsum('nij,nj->ni', Lc, sim_rng.standard_normal((len(sn), 3)))
    Ds = dict(dat)
    Ds['x1'], Ds['c'] = dat['x1'] + e3[:, 1], dat['c'] + e3[:, 2]
    Ds['mB'] = (truth_[0] + dat['mu'] - truth_[1] * dat['x1'] + truth_[2] * dat['c'] + truth_[3] * dat['step']
                + e3[:, 0] + np.exp(truth_[4]) * sim_rng.standard_normal(len(sn)) + np.sqrt(dat['var_pec']) * sim_rng.standard_normal(len(sn)))
    rs = minimize(lambda p: -lnpost(p, Ds), (-19.2, 0.1, 2.5, 0.0, np.log(0.1)), method='Nelder-Mead', options=dict(maxiter=40000, xatol=1e-8, fatol=1e-8))
    est.append(rs.x); err.append(np.sqrt(np.diag(np.linalg.inv(hessian(lambda p: -lnlike(p, Ds), rs.x)))))
est, err = np.array(est), np.array(err)
cover = np.mean(np.abs(est - truth_) < err, axis=0); pull = (est - truth_) / err
print("param        truth    mean est   mean sigma   coverage   mean pull +/- se")
for i, n in enumerate(names):
    print(f"{n:10s} {truth_[i]:9.3f} {est[:, i].mean():10.3f} {err[:, i].mean():11.4f} {cover[i]:10.2f} {pull[:, i].mean():+9.2f} +/- {pull[:, i].std() / np.sqrt(len(pull)):.2f}")

# why is beta low? control: same fake data but the regressors x1, c are NOT perturbed (their noise enters only through the m_B scatter)
est_c, err_c = [], []
for isim in range(200):
    e3 = np.einsum('nij,nj->ni', Lc, sim_rng.standard_normal((len(sn), 3)))
    Ds = dict(dat)
    Ds['mB'] = (truth_[0] + dat['mu'] - truth_[1] * dat['x1'] + truth_[2] * dat['c'] + truth_[3] * dat['step']
                + e3[:, 0] + truth_[1] * e3[:, 1] - truth_[2] * e3[:, 2]
                + np.exp(truth_[4]) * sim_rng.standard_normal(len(sn)) + np.sqrt(dat['var_pec']) * sim_rng.standard_normal(len(sn)))
    rs = minimize(lambda p: -lnpost(p, Ds), (-19.2, 0.1, 2.5, 0.0, np.log(0.1)), method='Nelder-Mead', options=dict(maxiter=40000, xatol=1e-8, fatol=1e-8))
    est_c.append(rs.x); err_c.append(np.sqrt(np.diag(np.linalg.inv(hessian(lambda p: -lnlike(p, Ds), rs.x)))))
est_c, err_c = np.array(est_c), np.array(err_c)
cover_c = np.mean(np.abs(est_c - truth_) < err_c, axis=0); pull_c = (est_c - truth_) / err_c
print("\ncontrol (regressors exact):  coverage", np.round(cover_c, 2), "  mean pull", np.round(pull_c.mean(axis=0), 2))

# size of the effect: regression dilution. Expected slope ratio = (1 + error-covariance term) / (1 + noise fraction of the regressor's variance), 1/variance weights
w_ = 1 / var_i(truth_, dat)
for nm_, x_, ex_, cmb_, coef_, i_ in (('c', dat['c'], dat['c_err'], dat['cov_mB_c'], truth_[2], 2), ('x1', dat['x1'], dat['x1_err'], dat['cov_mB_x1'], -truth_[1], 1)):
    S_ = np.sum(w_ * (x_ - np.average(x_, weights=w_))**2)
    noise, covterm = np.sum(w_ * ex_**2) / S_, np.sum(w_ * cmb_) / (coef_ * S_)
    print(f"{nm_}: noise fraction {noise:.3f}, error-covariance term {covterm:+.3f} -> expected coefficient ratio {(1 + covterm) / (1 + noise):.3f};  simulated {est[:, i_].mean() / truth_[i_]:.3f}")
fig, ax = plt.subplots(1, 5, figsize=(15, 2.8))
xg = np.linspace(-4, 4, 200)
for i_, n_ in enumerate(names):
    ax[i_].hist(pull[:, i_], 20, range=(-4, 4), density=True, color='0.7', label='fakes with noisy $x_1$, $c$'); ax[i_].hist(pull_c[:, i_], 20, range=(-4, 4), density=True, histtype='step', color='C0', label='control')
    ax[i_].plot(xg, stats.norm.pdf(xg), 'k', lw=0.7); ax[i_].set_xlabel(f'pull of {n_}')
ax[0].legend(fontsize=6); plt.tight_layout(); plt.show()
plt.figure(figsize=(5.5, 3)); xb = np.arange(5)
plt.bar(xb - 0.2, cover, 0.4, label='noisy $x_1$, $c$'); plt.bar(xb + 0.2, cover_c, 0.4, label='control')
plt.axhspan(0.60, 0.76, color='C2', alpha=0.2, label='criterion'); plt.axhline(0.68, color='k', lw=0.6)
plt.xticks(xb, names, fontsize=8); plt.ylabel('68% coverage'); plt.legend(fontsize=7); plt.tight_layout(); plt.show()

<div style="color:#1f6feb">

**ANSWER:**


The result against the criterion I wrote above is that **my pipeline fails it, for $\beta$** (and by the second criterion for $M$, $\alpha$ and $\gamma$ too).

| parameter | truth | mean estimate | 68% coverage (200 sets) | mean pull |
|---|---|---|---|---|
| $M$ | $-19.300$ | $-19.303$ | $0.62$ | $-0.25 \pm 0.07$ |
| $\alpha$ | $0.140$ | $0.136$ | $0.66$ | $-0.51 \pm 0.07$ |
| $\beta$ | $2.90$ | $2.680$ | $0.06$ | $-2.35 \pm 0.07$ |
| $\gamma$ | $-0.060$ | $-0.052$ | $0.64$ | $+0.46 \pm 0.07$ |
| $\ln\sigma_{\rm int}$ | $-2.21$ | $-2.21$ | $0.66$ | $-0.01 \pm 0.07$ |

Both criteria have to pass for the pipeline to pass. Coverage is fine for $M$, $\alpha$, $\gamma$ and $\sigma_{\rm int}$ (0.62-0.66, inside 0.60-0.76) and fails badly for $\beta$ (0.06). The mean-pull criterion trips for $M$, $\alpha$, $\beta$ and $\gamma$. For $M$, $\alpha$ and $\gamma$ the pulls are small ($-0.25\sigma$, $-0.5\sigma$ and $+0.5\sigma$), so I read those as small biases and not a broken pipeline. $\beta$ is the one the coverage test can't forgive.

Why it fails: this is attenuation (regression dilution), and the likelihood itself is fine. In my fake data the observed $x_1$ and $c$ have their own measurement noise, and the fit treats them as exact regressors. Noise in a regressor pulls its coefficient toward zero. To check that, I ran a control where the regressors are exact and their noise only enters through the $m_B$ scatter. The coverage there is 0.62-0.73 and the pulls are $\le 0.11$ for $M$, $\alpha$, $\beta$ and $\gamma$, so the same code recovers the truth. ($\ln\sigma_{\rm int}$ has a pull of $-0.27$ in the control, with coverage 0.66, inside the coverage criterion. The pull alone would trip the pull criterion. My guess is that it's the usual downward bias of a maximum-likelihood variance, which would be about $-0.14$ for 5 parameters and 246 points, but I didn't test that.)

How big is the effect? The noise fraction is the share of a regressor's (weighted) variance that is measurement noise, and the error-covariance term is how far the correlated errors on $m_B$ and the regressor push the slope back the other way. The expected ratio of the fitted to the true coefficient is $(1 + \text{error-covariance term})/(1 + \text{noise fraction})$, computed from this file's own errors. For $\beta$ it's $0.922$ and the simulation gives $0.924$. For $\alpha$ it's $0.949$ against $0.969$ simulated, so that one only roughly agrees. The noise fraction alone would predict $-15\%$ for $\beta$, but the errors on $m_B$ and $c$ are positively correlated, and that cancels about half of it. I worked this out after the surprise, to explain it.

What it means for the science: $\beta$ comes out about 8% low and $\alpha$ about 3% low. A latent-variable model, where the true $x_1$ and $c$ are extra parameters, would remove that. $\gamma$ shifts by $+0.008$ mag ($0.5\sigma$) through its correlations with $M$ and $\alpha$, which is small against $\sigma_\gamma = 0.019$, so the 2d answer stays "weak". I wrote the criterion before the run and I didn't change it after seeing the result.

</div>

**Points (8):** 2 for a test that could fail, with the failure criterion written before you run it. 3 for running it properly (30 or more fake data sets, your own redshifts, stretches and colors, a $\gamma$ you chose, the same pipeline). 2 for reporting the result against your criterion. 1 for an honest reading of the result.

## Part 3: The referee report (25 pts)

White dwarfs are the cores of dead stars like the Sun. Your file `p3_wd_masses.csv` holds 600 real DA (hydrogen-atmosphere) white dwarfs from the 100 pc sample of [Kilic et al. (2020)](https://arxiv.org/abs/2006.00323), ApJ 898, 84: `name`, `teff_K`, `mass_Msun`, `mass_err_Msun`. Most single white dwarfs have masses near $0.6\,M_\odot$; Kilic et al. report a narrow peak there plus an excess of more massive ones, around $0.7$-$0.9\,M_\odot$, which merger products alone do not explain.

The notebook `labs/midterm/ai_solution.ipynb` fits your masses with a two-population model using `emcee`. Run it from that directory, after setting `NETID` in its first cell. It runs top to bottom and the plots look nice, but it's wrong in at least three substantive ways - and if you don't look at your data, you'll fool yourself the same way.

### 3a - The report (20 pts)

1. **Identify** at least three substantive errors (not style; "it used 32 walkers and I'd use 48" is not one).
2. **Demonstrate** each on *your* file: show, with a number or a plot, what the error does to the answer. A demonstration can come out negative: if an error doesn't move your numbers, show that, and say what it would take for it to matter.
3. **State** the correct treatment, and run it where you can.

In [ ]:
# 3a: your work here
wd = load('p3_wd_masses'); m_all, e_all = wd['mass_Msun'].to_numpy(), wd['mass_err_Msun'].to_numpy()
npdf = lambda x, mu, s: np.exp(-0.5 * ((x - mu) / s)**2) / (s * np.sqrt(2 * np.pi))

def lnl(p, m, e=None, ordered=False):
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1): return -np.inf
    if ordered and mu1 >= mu2: return -np.inf
    e = np.zeros_like(m) if e is None else e
    return np.sum(np.log(f * npdf(m, mu1, np.hypot(s1, e)) + (1 - f) * npdf(m, mu2, np.hypot(s2, e))))

def mapfit(m, e=None, ordered=False, p0=(0.59, 0.03, 0.72, 0.17, 0.5)):
    r = minimize(lambda p: -lnl(p, m, e, ordered) if np.isfinite(lnl(p, m, e, ordered)) else 1e10, p0, method='Nelder-Mead', options=dict(maxiter=20000, xatol=1e-8, fatol=1e-8))
    return r.x

# the AI's pipeline, reproduced
med_, sd_ = np.median(m_all), np.std(m_all)
keep = np.abs(m_all - med_) < 3 * sd_
m_clip = m_all[keep]
lo_, hi_ = m_clip[m_clip < 0.7], m_clip[m_clip >= 0.7]
p0_ai = np.array([lo_.mean(), lo_.std(), hi_.mean(), hi_.std(), lo_.size / m_clip.size])

print(f"ERROR 1 - the 3-sigma clip removes {np.sum(~keep)} stars, all at M = {np.sort(m_all[~keep])}")
def mapbest(m, starts=(p0_ai, (0.59, 0.03, 0.73, 0.2, 0.45))):          # the mixture has local modes: keep the better of two starts
    fits = [mapfit(m, p0=st) for st in starts]
    print('   ln L at the two starts:', [round(lnl(p, m), 1) for p in fits], ' (the better one is kept)')
    return max(fits, key=lambda p: lnl(p, m))
pc, pu = mapbest(m_clip), mapbest(m_all)
print("                 mu1     sigma1  mu2     sigma2  f")
print("with the clip   ", np.round(pc, 3)); print("without the clip", np.round(pu, 3))
bins_ = np.linspace(m_all.min(), m_all.max(), 60)
plt.figure(figsize=(8, 3.2)); plt.hist(m_all, bins_, color='0.7', label='all 600'); plt.hist(m_all[~keep], bins_, color='C3', label=f'removed by the clip ({np.sum(~keep)})')
plt.axvline(med_ + 3 * sd_, color='C3', lw=0.8, ls='--'); plt.xlabel('mass ($M_\\odot$)'); plt.ylabel('count'); plt.yscale('log'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ERROR 2 - R-hat across the walkers of one ensemble is not a convergence test
def gelman_rubin(x):                                  # x: (steps, chains), as in the AI's notebook
    n, mc = x.shape
    B = n * np.var(x.mean(axis=0), ddof=1); Wv = np.mean(np.var(x, axis=0, ddof=1))
    return np.sqrt(((n - 1) / n * Wv + B / n) / Wv)

def run_ens(p_start, m, e=None, ordered=False, nwalk=32, nstep=3000, spread=1e-3, rel=False, seed=0):
    r_ = np.random.default_rng(seed)
    st = p_start * (1 + spread * r_.standard_normal((nwalk, 5))) if rel else p_start + spread * r_.standard_normal((nwalk, 5))
    s_ = emcee.EnsembleSampler(nwalk, 5, lambda p: lnl(p, m, e, ordered)); s_.random_state = np.random.RandomState(seed).get_state(); s_.run_mcmc(st, nstep, progress=False)
    return s_.get_chain(discard=500)                 # (steps, walkers, params)

pn = ['mu1', 'sigma1', 'mu2', 'sigma2', 'f']
ai_runs = [run_ens(p0_ai, m_clip, seed=s_) for s_ in range(12)]            # the AI's check repeated: its sampler is not seeded, so every run of its notebook differs
ai_max = np.array([max(gelman_rubin(c_[:, :, i]) for i in range(5)) for c_ in ai_runs])
print("the AI's own check (32 walkers, one 1e-3 ball): max R-hat over the five parameters in 12 repeats:", np.round(ai_max, 2))
print(f"fraction that print 'converged' (all < 1.1): {np.mean(ai_max < 1.1):.2f}")
ens0 = ai_runs[int(np.argmax(ai_max < 1.1))]                                 # first repeat that passes, used for the plots and the AI's quoted numbers
print("a run that passes:", {n: round(gelman_rubin(ens0[:, :, i]), 3) for i, n in enumerate(pn)})
swap = np.array([p0_ai[2], p0_ai[3], p0_ai[0], p0_ai[1], 1 - p0_ai[4]])                  # same solution, labels swapped
starts_ai = [p0_ai, swap, np.array([0.55, 0.05, 0.9, 0.2, 0.7]), np.array([0.65, 0.1, 0.8, 0.1, 0.3])]
ens4 = [run_ens(s_, m_clip, seed=2 + i, spread=2e-2, rel=True) for i, s_ in enumerate(starts_ai)]
allw = np.concatenate(ens4, axis=1)
print("R-hat across 4 dispersed ensembles (one label-swapped):", {n: round(gelman_rubin(allw[:, :, i]), 2) for i, n in enumerate(pn)})
print("mu1 median per ensemble:", [round(float(np.median(c_[:, :, 0])), 3) for c_ in ens4])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for k_, c_ in enumerate(ens4): ax[1].plot(c_[:, 0, 0], lw=0.4, label=f'ensemble {k_ + 1}' + (' (labels swapped)' if k_ == 1 else ''))
ax[0].plot(ens0[:, :, 0], lw=0.3, alpha=0.6); ax[0].set_title("the AI's check: 32 walkers, one ball", fontsize=9); ax[1].set_title('four dispersed ensembles', fontsize=9)
for a_ in ax: a_.set_xlabel('step'); 
ax[0].set_ylabel('$\\mu_1$ ($M_\\odot$)'); ax[1].legend(fontsize=7); plt.tight_layout(); plt.show()

In [ ]:
# CORRECT TREATMENT: no clip, per-star errors convolved, ordered means, four dispersed ensembles
cen = [np.array([0.56, 0.03, 0.80, 0.25, 0.30]), np.array([0.62, 0.08, 0.95, 0.15, 0.70]), np.array([0.58, 0.05, 0.70, 0.30, 0.50]), np.array([0.60, 0.02, 0.85, 0.10, 0.40])]
hon = [run_ens(c_, m_all, e_all, ordered=True, nstep=6000, spread=2e-2, rel=True, seed=20 + i) for i, c_ in enumerate(cen)]
allh = np.concatenate(hon, axis=1)                                      # (steps, walkers x ensembles, params)
idh = az.from_dict({'posterior': {n: allh[:, :, i].T for i, n in enumerate(pn)}})
sh = az.summary(idh); print(sh[['mean', 'sd', 'ess_bulk', 'r_hat']])
fl = allh.reshape(-1, 5); qq = np.percentile(fl, [16, 50, 84], axis=0)
for i, n in enumerate(pn): print(f"{n:7s} = {qq[1, i]:.4f} +{qq[2, i] - qq[1, i]:.4f} -{qq[1, i] - qq[0, i]:.4f}")
p_ai_best = np.median(ens0.reshape(-1, 5), axis=0)
smu2 = 0.5 * (qq[2, 2] - qq[0, 2])                      # half the 16-84 range of the posterior of mu2
print(f"\nERROR 3 - the AI quotes mu2 = {p_ai_best[2]:.3f} +/- {p_ai_best[3]:.3f} (the population width); the posterior sigma(mu2) is {smu2:.4f}: {p_ai_best[3] / smu2:.0f}x smaller")
xm = np.linspace(0.3, 1.35, 500); pm = qq[1]
c1_ = pm[4] * npdf(xm, pm[0], pm[1]); c2_ = (1 - pm[4]) * npdf(xm, pm[2], pm[3])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].hist(m_all, 60, density=True, color='0.85'); ax[0].plot(xm, c1_, label='narrow'); ax[0].plot(xm, c2_, label='broad'); ax[0].plot(xm, c1_ + c2_, 'k', label='total (errors not shown)')
ax[0].set_xlabel('mass ($M_\\odot$)'); ax[0].legend(fontsize=8); ax[0].set_yscale('log'); ax[0].set_ylim(1e-2, 20)
ax[1].hist(fl[:, 2], 60, density=True, color='C0', alpha=0.6, label=f'posterior of $\\mu_2$ ($\\sigma$ = {smu2:.3f})')
xa = np.linspace(p_ai_best[2] - 3 * p_ai_best[3], p_ai_best[2] + 3 * p_ai_best[3], 400)
ax[1].plot(xa, npdf(xa, p_ai_best[2], p_ai_best[3]), 'C3', label=f'AI "$\\pm$": {p_ai_best[3]:.3f}'); ax[1].set_xlabel('$\\mu_2$ ($M_\\odot$)'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# bonus errors 4-6
# 4: the validation cannot fail. Simulate from the AI's best fit (no errors, no clip), refit the same way.
rs_ = np.random.default_rng(5); bf = p_ai_best; rec = []
for _ in range(100):
    n2_ = rs_.binomial(m_clip.size, 1 - bf[4])
    sim = np.r_[rs_.normal(bf[0], bf[1], m_clip.size - n2_), rs_.normal(bf[2], bf[3], n2_)]
    rec.append(mapfit(sim, p0=bf))
rec = np.array(rec)
print("4. circular validation (100 AI-style realizations): mean recovered / input =", np.round(rec.mean(axis=0) / bf, 3), " scatter/input =", np.round(rec.std(axis=0) / bf, 3))
rec2 = []
for _ in range(100):                                   # a realistic test: add the per-star errors and apply the 3-sigma clip
    n2_ = rs_.binomial(m_all.size, 1 - bf[4]); s_i = np.r_[rs_.normal(bf[0], bf[1], m_all.size - n2_), rs_.normal(bf[2], bf[3], n2_)]
    s_i = s_i + e_all * rs_.standard_normal(m_all.size); s_i = s_i[np.abs(s_i - np.median(s_i)) < 3 * np.std(s_i)]
    rec2.append(mapfit(s_i, p0=bf))
rec2 = np.array(rec2); print("   realistic test (errors + clip):  mean recovered / input =", np.round(rec2.mean(axis=0) / bf, 3))
# 5: per-star errors ignored: how much does sigma1 move?
pe, pn_ = mapfit(m_all, e_all, ordered=True), mapfit(m_all, ordered=True)
print(f"5. sigma1 with errors convolved {pe[1]:.3f}, errors ignored {pn_[1]:.3f};  median error {np.median(e_all):.3f} Msun;  means with / without errors: mu1 {pe[0]:.3f} / {pn_[0]:.3f}, mu2 {pe[2]:.3f} / {pn_[2]:.3f}")
# 6: raw Delta lnL vs a penalized comparison
lnL1_ = np.sum(stats.norm.logpdf(m_all, m_all.mean(), m_all.std()))
print(f"6. Delta lnL (2 comp - 1 comp, no errors) = {lnl(pn_, m_all, None, True) - lnL1_:.1f};  Delta BIC = {2 * (lnl(pn_, m_all, None, True) - lnL1_) - 3 * np.log(m_all.size):.0f};  sigma2 = {pe[3]:.3f} Msun")
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
ax[0].plot(m_all, e_all, '.', ms=3, color='0.5'); ax[0].axhline(pe[1], color='C3', lw=0.8, label='$\\sigma_1$ (narrow width)'); ax[0].set_xlabel('mass ($M_\\odot$)'); ax[0].set_ylabel('quoted error ($M_\\odot$)'); ax[0].legend(fontsize=8)
for a_, i_, nm_ in zip(ax[1:], (1, 3), ('$\\sigma_1$', '$\\sigma_2$')):
    a_.hist(rec[:, i_] / bf[i_], 20, color='0.7', label="AI-style (its own model)"); a_.hist(rec2[:, i_] / bf[i_], 20, histtype='step', color='C3', label='with errors + clip'); a_.axvline(1, color='k', lw=0.6)
    a_.set_xlabel(f'recovered / input {nm_}'); a_.legend(fontsize=7)
plt.tight_layout(); plt.show()

<div style="color:#1f6feb">

**REFEREE REPORT:**


My recommendation is not to accept it as it stands. The notebook fits a two-Gaussian mixture, but most of its central claims don't hold up when I rerun them on this file (the numbers below are for these 600 stars).

**Major errors, each shown on this file**

1. The $3\sigma$ clip removes the massive tail. It drops 10 stars, all between $1.16$ and $1.30\ M_\odot$, and that's the tail whose existence is the question. The standard deviation it clips with is inflated by those same stars. If I refit the same model without the clip, $\mu_2$ goes from $0.708$ to $0.727$ (a shift of $1.5$ times the $\sigma(\mu_2)$ I get at the end), $\sigma_2$ from $0.201$ to $0.219$ and $f$ from $0.421$ to $0.435$. The fix is to not clip. A mixture's second component already fits a tail, which is also why a broad second component is hard to read as a population (see error 6). If outliers have to be modeled, give them their own component.
2. $\hat R$ across the walkers of one ensemble isn't a convergence test. All 32 walkers start in one $10^{-3}$ ball, so they can only tell us about that neighborhood. The AI's check is also noisy: its sampler is unseeded, so every run of its notebook gives different numbers, and it prints "All R-hat < 1.1: the chains have converged" whatever they are. In 12 repeats the largest $\hat R$ runs from $1.01$ to $4.6$, and only 6 of the 12 would pass its own $< 1.1$ test. Even a run that passes ($\hat R \le 1.07$) says nothing about a second mode. With unordered means a mixture has two equivalent solutions, because the labels can swap, and one tight ball can't reveal that. Four ensembles started far apart, one of them with the labels swapped, give $\hat R = 3.4$ ($\mu_1$), $6.0$ ($\sigma_1$), $2.9$ ($\mu_2$), $5.4$ ($\sigma_2$) and $1.7$ ($f$), and the $\mu_1$ medians are $0.594$, $0.708$ (the swapped one), $0.593$ and $0.594$. The fix is to order the means ($\mu_1 < \mu_2$) and to judge convergence across ensembles that start far apart.
3. A population width is quoted as an uncertainty. The report says $\mu_2 = 0.708 \pm 0.202\ M_\odot$, but $0.202$ is the width of the second population. The uncertainty on $\mu_2$ from the posterior is $0.0129\ M_\odot$, 16 times smaller. The same goes for every "$\pm$" in its results. It's the mirror image of 1d(iii): there $\sigma/\sqrt N$ was too small because it measured the average of the samples, and here a population width is used as the error on a mean, which makes it too big.

**Further errors**

4. The validation can't fail. It draws one data set from the fitted model with no errors and no clip, starts the sampler at the truth, and prints "validated". Over 100 such realizations the recovered parameters are within 3% of the input, which is what you'd expect when the test is built from the answer. A realistic test (add the file's per-star errors, apply the same $3\sigma$ clip, then run the AI's fit) is off by $+5.7\%$ in $\sigma_1$, $-7.7\%$ in $\sigma_2$ and $-5.1\%$ in $f$.
5. The per-star mass errors are ignored. It says they are "about 0.01", and the median is $0.011\ M_\odot$. Adding them in quadrature to each width moves $\sigma_1$ from $0.037$ to $0.033$ ($-11\%$) and leaves the means alone ($\mu_1$ is $0.593$ against $0.594$, and $\mu_2$ is $0.726$ against $0.727$). It makes a difference, but a small one, and it matters mostly for $\sigma_1$.
6. The "decisive" second population rests on a raw $\Delta\ln L$ ($129.0$) with no penalty for the extra parameters. $\Delta\mathrm{BIC} = 239$, so the conclusion is still right but the evidence statement is weak. The bigger problem is $\sigma_2 = 0.219\ M_\odot$: a component that broad is fitting the background, not the $0.7$-$0.9\ M_\odot$ shoulder the authors need.

**The correct treatment, run on this file.** I don't clip, I add the per-star errors in quadrature to each width, and I order the means ($\mu_1 < \mu_2$). I ran four ensembles of 32 walkers for 6000 steps, started far apart. Convergence is $\hat R \le 1.03$ for all five parameters (I treat each walker as a chain, so this checks that the ensembles agree - it doesn't make the walkers independent). The results are $\mu_1 = 0.5935 \pm 0.0033$, $\sigma_1 = 0.0332 \pm 0.0035$, $\mu_2 = 0.726 \pm 0.013$, $\sigma_2 = 0.220 \pm 0.009$ and $f = 0.435 \pm 0.033$ (the weight of the narrow component), all in $M_\odot$ except $f$. I compare models with BIC and not with a raw $\Delta\ln L$.

</div>

**Points (20):** 6 for naming at least three substantive errors (2 each - style complaints like the number of walkers are 0). 9 for showing each one on your own file with a number or a plot (3 each - an assertion with no demonstration is 1). 5 for the correct treatment, run where you can (1 each for five of these: no clip, ordered means, dispersed chains, per-star errors, the posterior $\sigma(\mu)$ as the error bar, a penalized comparison).

### 3b - AI-use appendix (5 pts)

1. Which tools did you use, for what, what did they get wrong, and how did you catch it? "I used ChatGPT to help" is useless; "none" is fine if that's true. I promise I don't care which - just be honest about what you did.
2. What would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing, in two or three sentences.

<div style="color:#1f6feb">

**AI-USE APPENDIX:**


*This is an example of what a good appendix looks like. The author here is the assistant that helped write this solution set.*

1. **Tool:** Claude Code (Claude Sonnet 5.5). It wrote the code and the first draft of each answer, and ran them. **What it got wrong, and how I caught it:** (i) My first version of the 3a clip comparison started the optimizer at the AI notebook's own split-at-0.7 guess. That start ends up in a poor local mode ($\ln L = 250$ against $339$ - the cell now prints both) and gave $\mu_2 = 0.925$, $\sigma_2 = 0.106$, $f = 0.84$ for the clipped data. I caught it because the log-likelihood differed by 89 between starts, and because the result didn't match the fit from the better start. The fix is to keep the better of two starts. (ii) I had assumed from the problem statement that the contaminants would be hard to see. Printing the ranks showed that four of them are $5.3$-$6.1\sigma$ outliers after a plain regression, so I rewrote the 2a answer to say what the data show. (iii) My first guess for the $\beta$ bias in 2e was the dilution fraction alone ($-15\%$). The simulation gave about $-8\%$. The piece I was missing was the covariance between the $m_B$ and $c$ errors, and the control simulation is what showed it. (iv) A first draft said the AI's $\hat R$ was "close to 1" because the one run I looked at gave about $1.01$. Twelve repeats show that it ranges from $1.01$ to $4.6$ and passes the AI's own $< 1.1$ test only 6 times in 12. I caught that by repeating the run with different seeds.
2. **What would have gone wrong if I hadn't checked:** in 3a I would have told the referee that removing the clip widens the second population from $0.106$ to $0.219\ M_\odot$ and moves its mean by $0.20\ M_\odot$. That comes from a bad starting point (the clip itself only moves the mean by $0.019\ M_\odot$), and the referee report would have overstated error 1 by a factor of about 10.

</div>

**Points (5):** 3 for the tools, what each was used for, what it got wrong and how you caught it ("I used ChatGPT to help" is 1, and "none" is full credit if it's true). 2 for one specific thing that would have gone wrong if you hadn't checked.

Get started. You know how to do all of this. If not, there's something in a lab or class notebook that can help. Slack me with questions. Best of luck.